# 12.03 · LoRA & QLoRA

**Chapter:** Fine-tuning & Post-training · **Milestone:** M6

## Goals
- [ ] LoRA in one equation; rank, alpha, target modules; why it's cheap
- [ ] Fine-tune on a Mac two ways: **HF TRL + PEFT** (PyTorch/MPS) and **MLX-LM** (Apple-native)
- [ ] Evaluate before/after on in-distribution **and** held-out templates
- [ ] Know the GPU path: QLoRA with Unsloth on free Colab/Kaggle, plus Axolotl, LLaMA-Factory, managed fine-tuning APIs

## Resources
- [LoRA paper](https://arxiv.org/abs/2106.09685) · [QLoRA paper](https://arxiv.org/abs/2305.14314) — free
- [HF PEFT docs](https://huggingface.co/docs/peft/) · [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) — free
- [Unsloth notebooks (Colab/Kaggle, free GPUs)](https://github.com/unslothai/notebooks) · [MLX-LM LoRA guide](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md) — free

| Tool | Hardware | Notes |
|---|---|---|
| **TRL + PEFT** | CUDA, Apple MPS, CPU | the reference stack; SFT, DPO, GRPO all in one library |
| **MLX-LM** | Apple Silicon | fastest on Macs; CLI `mlx_lm lora` |
| **Unsloth** | NVIDIA GPU (free Colab T4 / Kaggle) | ~2× faster, much less memory; great free notebooks |
| **Axolotl**, **LLaMA-Factory** | NVIDIA GPUs | YAML configs, multi-GPU, many recipes |
| **Managed APIs** | none (paid) | OpenAI, Together, Fireworks, Google Vertex: upload JSONL, get a model endpoint |

In [ ]:
import json
import subprocess
import sys
import time

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

from aieng import DATA
from task import dataset, messages, score

FT = DATA / "ft"; FT.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"

## 1. LoRA in one equation
Freeze the pretrained weight $W$ ($d \times k$). Learn a low-rank update: $W' = W + \frac{\alpha}{r} BA$ with $B$ ($d \times r$), $A$ ($r \times k$),
$r \ll d$. Only $A, B$ train → tiny optimizer state, tiny checkpoint ("adapter"), swap adapters per task. (SVD intuition: 01.03.)

**QLoRA**: also store the frozen $W$ in 4-bit (NF4) → a 7B model fine-tunes on a 16 GB GPU. Needs `bitsandbytes` + CUDA (not on Mac).

In [ ]:
d, k, r = 4096, 4096, 16
print(f"full update: {d * k:,} params | LoRA r={r}: {r * (d + k):,} params ({r * (d + k) / (d * k):.2%})")

## 2. Data (same generator and splits as 12.02, regenerated here so this notebook runs on its own)
Train on 6 of 7 English templates + 3 of 4 Spanish; test on the same templates (in-distribution) and on held-out ones (OOD).

In [ ]:

HELD_OUT_PREFIXES = ("Netflix charged me", "me cobraron")
pool = dataset(4000, seed=1)
train_rows = [e for e in pool if not e["text"].startswith(HELD_OUT_PREFIXES)][:1500]
test_in = [e for e in dataset(400, seed=2) if not e["text"].startswith(HELD_OUT_PREFIXES)][:60]
test_ood = [e for e in dataset(800, seed=3) if e["text"].startswith(HELD_OUT_PREFIXES)][:60]

train_ds = Dataset.from_list([{"prompt": messages(e["text"]), "completion": [{"role": "assistant", "content": json.dumps(e["target"])}]}
                              for e in train_rows])  # prompt-completion → loss on the answer only (12.02)
print(train_ds, len(test_in), len(test_ood))

## 3. Path A — TRL + PEFT on PyTorch (MPS)

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)


def generate(model, msgs, max_new_tokens=48):
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)


def evaluate(gen_fn, rows):
    s = [score(gen_fn(messages(e["text"])), e["target"]) for e in rows]
    return {k: round(sum(x[k] for x in s) / len(s), 2) for k in ("valid_json", "exact")}


model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(device)
before = {"in-dist": evaluate(lambda m: generate(model, m), test_in), "held-out": evaluate(lambda m: generate(model, m), test_ood)}
print("before:", before)

In [ ]:
config = SFTConfig(
    output_dir=str(FT / "smollm-sft-run"),
    max_steps=150,  # ~1.6 epochs of 1,500 examples at batch 16
    per_device_train_batch_size=16,
    learning_rate=2e-3,  # LoRA tolerates much higher LRs than full fine-tuning (~1e-5)
    lr_scheduler_type="cosine", warmup_steps=10,
    logging_steps=25, save_strategy="no", report_to="none",
    max_length=256,
)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM")
trainer = SFTTrainer(model=model, args=config, train_dataset=train_ds, processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()

start = time.perf_counter()
trainer.train()
print(f"trained in {time.perf_counter() - start:.0f}s on {device}")
losses = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
print("loss by step:", losses)

In [ ]:
ft_model = trainer.model.eval()
after = {"in-dist": evaluate(lambda m: generate(ft_model, m), test_in), "held-out": evaluate(lambda m: generate(ft_model, m), test_ood)}
print("before:", before)
print("after :", after)
for e in test_ood[:3]:
    print(f"  {e['text']!r}\n    -> {generate(ft_model, messages(e['text']))}")

trainer.model.save_pretrained(FT / "smollm-lora")  # adapter only: a few MB
tok.save_pretrained(FT / "smollm-lora")
print("adapter files:", sorted(p.name for p in (FT / "smollm-lora").iterdir()))

A 135M model at near-perfect exact-match after ~3 minutes on a laptop. Check the held-out templates: generalising to unseen phrasings is the real
test. If held-out lags in-distribution, the fix is more diverse training data (12.02 §5), not more epochs.

## 4. Path B — MLX-LM on Apple Silicon (Qwen3-0.6B)
Same data as chat `messages` JSONL; the CLI handles tokenisation, LoRA and checkpoints. `--mask-prompt` = loss on the answer only.

In [ ]:
import platform

mlx_dir = FT / "mlx"
(mlx_dir / "data").mkdir(parents=True, exist_ok=True)
for name, rows in [("train", train_rows[:800]), ("valid", test_in[:40])]:
    (mlx_dir / "data" / f"{name}.jsonl").write_text("\n".join(json.dumps({"messages": messages(e["text"], e["target"])}) for e in rows))

if platform.machine() == "arm64" and platform.system() == "Darwin":
    cmd = [sys.executable, "-m", "mlx_lm", "lora", "--model", "Qwen/Qwen3-0.6B", "--train", "--data", str(mlx_dir / "data"),
           "--iters", "80", "--batch-size", "8", "--learning-rate", "1e-4", "--mask-prompt",
           "--adapter-path", str(mlx_dir / "adapters"), "--steps-per-report", "20", "--steps-per-eval", "80"]
    start = time.perf_counter()
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "loss" in l.lower())[-600:], f"\n({time.perf_counter() - start:.0f}s)")

    from mlx_lm import generate as mlx_generate, load

    mlx_model, mlx_tok = load("Qwen/Qwen3-0.6B", adapter_path=str(mlx_dir / "adapters"))

    def mlx_gen(msgs):
        prompt = mlx_tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False)
        return mlx_generate(mlx_model, mlx_tok, prompt=prompt, max_tokens=60, verbose=False)

    print("MLX Qwen3-0.6B + LoRA:", {"in-dist": evaluate(mlx_gen, test_in[:30]), "held-out": evaluate(mlx_gen, test_ood[:30])})
else:
    print("MLX needs Apple Silicon — skip")

Then `python -m mlx_lm fuse --model Qwen/Qwen3-0.6B --adapter-path data/ft/mlx/adapters` merges the adapter into the weights (12.06).

## 5. Path C — QLoRA with Unsloth on a free GPU (Colab / Kaggle)
Copy into a Colab notebook (Runtime → T4 GPU) — same data, a 4-bit 1.7B–8B model:
```python
!pip install unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Qwen3-1.7B", max_seq_length=512, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds,
                     args=SFTConfig(per_device_train_batch_size=8, max_steps=200, learning_rate=2e-4, output_dir="out", report_to="none"))
trainer.train()
model.save_pretrained_gguf("payments-gguf", tokenizer, quantization_method="q4_k_m")   # straight to Ollama/llama.cpp
```
Axolotl (YAML) equivalent: `base_model`, `adapter: qlora`, `load_in_4bit: true`, `lora_r: 16`, `datasets: [{path: train.jsonl, type: chat_template}]`.
Managed: OpenAI `client.fine_tuning.jobs.create(training_file=..., model="gpt-4.1-mini-...")` — same JSONL `messages` format, paid per token.

## 6. Hyperparameters that matter
| Knob | Typical LoRA values | Notes |
|---|---|---|
| learning rate | 1e-4 – 2e-3 (small models tolerate higher) | the first thing to tune |
| rank `r` / `lora_alpha` | 8–64 / 1–2× r | higher rank = more capacity, rarely the bottleneck for narrow tasks |
| target modules | all linear layers | attention-only LoRA is weaker |
| epochs | 1–3 | watch validation loss; overfitting shows as perfect train, worse held-out |
| batch size × grad accumulation | 16–64 effective | memory vs stability |

## Exercises
1. Train with only 100, 300 and 1,500 examples. Plot held-out exact match vs dataset size. Where does it saturate?
2. Train on English templates only and test on Spanish ones. What does cross-lingual transfer look like in a 135M model?
3. Run Path C on Colab with Qwen3-1.7B in 4-bit, export GGUF, and compare its held-out score and speed with the 135M model.

In [ ]:
# your code here